# Hyperparameter tuning XGBOOST (death as a reference)

## 0. Package loading and installation

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1FMHIud9Hi0rIxnMqRfRFdzBQpKEzI796

In [1]:
# Commented out IPython magic to ensure Python compatibility.
# For Jupyter/Colab notebooks
%reset -f
import gc
gc.collect()

import numpy as np
import pandas as pd
import time

#conda install -c conda-forge \
#    numpy \
#    scipy \
#    pandas \
#    pyarrow \
#    scikit-survival \
#    spyder \
#    lifelines

# conda install -c conda-forge fastparquet
# conda install -c conda-forge xgboost
# conda install -c conda-forge pytorch cpuonly
# conda install -c pytorch pytorch cpuonly
# conda install -c conda-forge matplotlib
# conda install -c conda-forge seaborn
# conda install spyder-notebook -c spyder-ide
# conda install notebook nbformat nbconvert
# conda install -c conda-forge xlsxwriter
# conda install -c conda-forge shap

# import subprocess, sys

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "matplotlib"
# ])

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "seaborn"
# ])

print("numpy:", np.__version__)


from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    integrated_brier_score
)
from sksurv.util import Surv

#Dput
def dput_df(df, digits=6):
    data = {
        "columns": list(df.columns),
        "data": [
            [round(x, digits) if isinstance(x, (float, np.floating)) else x
             for x in row]
            for row in df.to_numpy()
        ]
    }
    print(data)


#Glimpse function
def glimpse(df, max_width=80):
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")
#Tabyl function
def tabyl(series):
    counts = series.value_counts(dropna=False)
    props = series.value_counts(normalize=True, dropna=False)
    return pd.DataFrame({"value": counts.index,
                         "n": counts.values,
                         "percent": props.values})
#clean_names
import re

def clean_names(df):
    """
    Mimic janitor::clean_names for pandas DataFrames.
    - Lowercase
    - Replace spaces and special chars with underscores
    - Remove non-alphanumeric/underscore
    """
    new_cols = []
    for col in df.columns:
        # lowercase
        col = col.lower()
        # replace spaces and special chars with underscore
        col = re.sub(r"[^\w]+", "_", col)
        # strip leading/trailing underscores
        col = col.strip("_")
        new_cols.append(col)
    df.columns = new_cols
    return df

numpy: 2.0.1


## Load data

In [2]:

from pathlib import Path

BASE_DIR = Path(
    r"G:\My Drive\Alvacast\SISTRAT 2023\data\20241015_out\pred1"
)

import pickle

with open(BASE_DIR / "imputations_list_jan26.pkl", "rb") as f:
    imputations_list_jan26 = pickle.load(f)

imputation_1 = pd.read_parquet(
    BASE_DIR / "imputation_1.parquet",
    engine="fastparquet"
)

In [3]:

import pandas as pd

for i in range(1, 6):
    globals()[f"imputation_nodum_{i}"] = pd.read_parquet(
        BASE_DIR / f"imputation_nondum_{i}.parquet",
        engine="fastparquet"
    )

In [4]:
from IPython.display import display, HTML
import io
import sys

def fold_output(title, func):
    buffer = io.StringIO()
    sys.stdout = buffer
    func()
    sys.stdout = sys.__stdout__
    
    html = f"""
    <details>
      <summary>{title}</summary>
      <pre>{buffer.getvalue()}</pre>
    </details>
    """
    display(HTML(html))


fold_output(
    "Show imputation_nodum_1 structure",
    lambda: imputation_nodum_1.info()
)

fold_output(
    "Show imputation_1 structure",
    lambda: imputation_1.info()
)

In [5]:
from IPython.display import display, Markdown

if isinstance(imputations_list_jan26, list) and len(imputations_list_jan26) > 0:
    display(Markdown(f"**First element type:** `{type(imputations_list_jan26[0])}`"))

    if isinstance(imputations_list_jan26[0], dict):
        display(Markdown(f"**First element keys:** `{list(imputations_list_jan26[0].keys())}`"))

    elif isinstance(imputations_list_jan26[0], (pd.DataFrame, np.ndarray)):
        display(Markdown(f"**First element shape:** `{imputations_list_jan26[0].shape}`"))

**First element type:** `<class 'pandas.DataFrame'>`

**First element shape:** `(88504, 56)`

This code block:

1.  **Imports the `pickle` library**: This library implements binary protocols for serializing and de-serializing a Python object structure.
2.  **Specifies the `file_path`**: It points to the `.pkl` file you selected.
3.  **Opens the file in binary read mode (`'rb'`)**: This is necessary for loading pickle files.
4.  **Loads the object**: `pickle.load(f)` reads the serialized object from the file and reconstructs it in memory.
5.  **Prints confirmation and basic information**: It verifies that the file was loaded and shows the type of the loaded object, and some details about the first element if it's a list containing common data structures.

### Format data

Due to inconsistencies and structural heterogeneity across previously merged datasets, we decided not to proceed with a direct inspection and comparison of column names between the first imputed dataset from `imputations_list_jan26` (which likely included dummy-encoded variables) and `imputation_nodum_1` (which likely retained non–dummy-encoded variables).

Instead, we reconstructed the analytic datasets *de novo* using the most recent source files available in the original directory (`BASE_DIR`). Time-to-event variables were re-derived to ensure internal consistency. Variables that could introduce information leakage (e.g., time from admission) were excluded, and the center identifier variable was removed prior to modeling.

In [6]:
#1.2. Build Surv objects from df_final
from IPython.display import display, Markdown
from sksurv.util import Surv

for i in range(1, 6):
    # Get the DataFrame
    df = globals()[f"imputation_nodum_{i}"]

    # Extract time and event arrays
    time_readm  = df["readmit_time_from_disch_m"].to_numpy()
    event_readm = (df["readmit_event"].to_numpy() == 1)
    time_death  = df["death_time_from_disch_m"].to_numpy()
    event_death = (df["death_event"].to_numpy() == 1)

    # Create survival objects
    y_surv_readm = Surv.from_arrays(event=event_readm, time=time_readm)
    y_surv_death = Surv.from_arrays(event=event_death, time=time_death)

    # Store in global variables (optional but matches your pattern)
    globals()[f"y_surv_readm_{i}"]  = y_surv_readm
    globals()[f"y_surv_death_{i}"]  = y_surv_death

    # Print info
    display(Markdown(f"\n--- Imputation {i} ---"))
    display(Markdown(
    f"**y_surv_readm dtype:** {y_surv_readm.dtype}  \n"
    f"**shape:** {y_surv_readm.shape}"
    ))
    display(Markdown(
    f"**y_surv_death dtype:** {y_surv_death.dtype}  \n"
    f"**shape:** {y_surv_death.shape}"
    ))



--- Imputation 1 ---

**y_surv_readm dtype:** [('event', '?'), ('time', '<f8')]  
**shape:** (88504,)

**y_surv_death dtype:** [('event', '?'), ('time', '<f8')]  
**shape:** (88504,)


--- Imputation 2 ---

**y_surv_readm dtype:** [('event', '?'), ('time', '<f8')]  
**shape:** (88504,)

...

**y_surv_readm dtype:** [('event', '?'), ('time', '<f8')]  
**shape:** (88504,)

**y_surv_death dtype:** [('event', '?'), ('time', '<f8')]  
**shape:** (88504,)


--- Imputation 5 ---

**y_surv_readm dtype:** [('event', '?'), ('time', '<f8')]  
**shape:** (88504,)

**y_surv_death dtype:** [('event', '?'), ('time', '<f8')]  
**shape:** (88504,)

In [7]:
fold_output(
    "Show imputation_nodum_1 (newer database) glimpse",
    lambda: glimpse(imputation_nodum_1)
)
fold_output(
    "Show first db of imputations_list_jan26 (older) glimpse",
    lambda: glimpse(imputations_list_jan26[0])
)


For each imputed dataset (1–5), we identified and removed predictors with zero variance, as they provide no useful information and can destabilize models. We printed the dropped variables and produced a cleaned version of each design matrix. This ensures that all downstream analyses use only informative predictors.

In [8]:
# Keep only these objects
objects_to_keep = {
    "objects_to_keep",
    "imputation_nodum_1",
    "imputation_nodum_2",
    "imputation_nodum_3",
    "imputation_nodum_4",
    "imputation_nodum_5",
    "y_surv_readm",
    "y_surv_death",
    "imputations_list_jan26"
}

import types

for name in list(globals().keys()):
    obj = globals()[name]
    if (
        name not in objects_to_keep
        and not name.startswith("_")
        and not callable(obj)
        and not isinstance(obj, types.ModuleType)  # <- protects ALL modules
    ):
        del globals()[name]

In [9]:
from IPython.display import display, Markdown

# 1. Define columns to exclude (same as before)
target_cols = [
    "readmit_time_from_disch_m",
    "readmit_event",
    "death_time_from_disch_m",
    "death_event",
]

leak_time_cols = [
    "readmit_time_from_adm_m",
    "death_time_from_adm_m",
]

center_id = ["center_id"]

cols_to_exclude = target_cols + center_id  + leak_time_cols

# 2. Create list of your EXISTING imputation DataFrames (1-5)
imputed_dfs = [
    imputation_nodum_1,
    imputation_nodum_2,
    imputation_nodum_3,
    imputation_nodum_4,
    imputation_nodum_5
]

# 3. Preprocessing loop
X_reduced_list = []

for d, df in enumerate(imputed_dfs):
    imputation_num = d + 1  # Convert 0-index to 1-index for display

    display(Markdown(f"\n=== Imputation dataset {imputation_num} ==="))

    # a) Identify and drop constant predictors
    const_mask = (df.nunique(dropna=False) <= 1)
    dropped_const = df.columns[const_mask].tolist()
    display(Markdown(f"**Constant predictors dropped ({len(dropped_const)}):**"))
    display(Markdown(f"{dropped_const if dropped_const else 'None'}"))

    # b) Remove constant columns
    X_reduced = df.loc[:, ~const_mask]

    # c) Drop target/leakage columns (if present)
    cols_to_drop = [col for col in cols_to_exclude if col in X_reduced.columns]
    if cols_to_drop:
        X_reduced = X_reduced.drop(columns=cols_to_drop)
        display(Markdown(f"**Dropped target/leakage columns:** {cols_to_drop}"))
    else:
        display(Markdown("No target/leakage columns found to drop"))

    # d) Store cleaned DataFrame
    X_reduced_list.append(X_reduced)

    # e) Report shapes
    display(Markdown(f"**Original shape:** {df.shape}"))
    display(Markdown(
        f"**Cleaned shape:** {X_reduced.shape} "
        f"(removed {df.shape[1] - X_reduced.shape[1]} columns)"
    ))

display(Markdown("\n✅ **Preprocessing complete! X_reduced_list contains 5 cleaned DataFrames.**"))


=== Imputation dataset 1 ===

**Constant predictors dropped (0):**

None

**Dropped target/leakage columns:** ['readmit_time_from_disch_m', 'readmit_event', 'death_time_from_disch_m', 'death_event', 'center_id', 'readmit_time_from_adm_m', 'death_time_from_adm_m']

**Original shape:** (88504, 43)

...

None

**Dropped target/leakage columns:** ['readmit_time_from_disch_m', 'readmit_event', 'death_time_from_disch_m', 'death_event', 'center_id', 'readmit_time_from_adm_m', 'death_time_from_adm_m']

**Original shape:** (88504, 43)

**Cleaned shape:** (88504, 36) (removed 7 columns)


✅ **Preprocessing complete! X_reduced_list contains 5 cleaned DataFrames.**

### Dummify
A structured preprocessing pipeline was implemented prior to modeling. Ordered categorical variables (e.g., housing status, educational attainment, clinical evaluations, and substance use frequency) were manually mapped to numeric scales reflecting their natural ordering. For nominal categorical variables, prespecified reference categories were enforced to ensure consistent baseline comparisons across imputations. All remaining categorical predictors were then converted to dummy variables using one-hot encoding with the first category dropped to prevent multicollinearity. The procedure was applied consistently across all imputed datasets to ensure harmonized model inputs.


In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype

def preprocess_features_robust(df):
    df_clean = df.copy()

    # ---------------------------------------------------------
    # 1. Ordinal encoding (your existing code)
    # ---------------------------------------------------------
    ordered_mappings = {
        # --- NEW: Housing & Urbanicity ---
        "tenure_status_household": {
            "illegal settlement": 4,                       # Situación Calle
            "stays temporarily with a relative": 3,        # Allegado
            "others": 2,                                   # En pensión / Otros
            "renting": 1,                                  # Arrendando
            "owner/transferred dwellings/pays dividends": 0 # Vivienda Propia
        },
        "urbanicity_cat": {
            "1.Rural": 2,
            "2.Mixed": 1,
            "3.Urban": 0
        },

        # --- Clinical Evaluations (Minimo -> Intermedio -> Alto) ---
        "evaluacindelprocesoteraputico": {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0},
        "eva_consumo":      {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0},
        "eva_fam":          {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0},
        "eva_relinterp":    {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0},
        "eva_ocupacion":    {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0},
        "eva_sm":           {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0},
        "eva_fisica":       {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0},
        "eva_transgnorma":  {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0},

        # --- Frequency (Less freq -> More freq) ---
        "prim_sub_freq_rec": {
            "1.≤1 day/wk": 0,
            "2.2–6 days/wk": 1,
            "3.Daily": 2
        },

        # --- Education (Less -> More) ---
        "ed_attainment_corr": {
            "3-Completed primary school or less": 2,
            "2-Completed high school or less": 1,
            "1-Completed higher education": 0
        }
    }

    for col, mapping in ordered_mappings.items():
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.strip()
            df_clean[col] = df_clean[col].map(mapping)

            n_missing = df_clean[col].isnull().sum()
            if n_missing > 0:
                if n_missing == len(df_clean):
                    print(f"⚠️ WARNING: Mapping failed completely for '{col}'.")
                mode_val = df_clean[col].mode()[0]
                df_clean[col] = df_clean[col].fillna(mode_val)

    # ---------------------------------------------------------
    # 2. FORCE reference categories for dummies
    # ---------------------------------------------------------
    dummy_reference = {
        "sex_rec": "man",
        "plan_type_corr": "ambulatory",
        "marital_status_rec": "married/cohabiting",
        "cohabitation": "alone",
        "sub_dep_icd10_status": "hazardous consumption",
        "tr_outcome": "completion",
        "adm_motive": "spontaneous consultation",
        "tipo_de_vivienda_rec2": "formal housing",
        "plan_type_corr": "pg-pab",
        "occupation_condition_corr24": "employed",
        "any_violence": "0.No domestic violence/sex abuse",
        "first_sub_used": "marijuana",
        }

    for col, ref in dummy_reference.items():
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.strip()
            cats = df_clean[col].unique().tolist()

            if ref in cats:
                new_order = [ref] + [c for c in cats if c != ref]
                cat_type = CategoricalDtype(categories=new_order, ordered=False)
                df_clean[col] = df_clean[col].astype(cat_type)
            else:
                print(f"⚠️ Reference '{ref}' not found in {col}")

    # ---------------------------------------------------------
    # 3. One-hot encoding
    # ---------------------------------------------------------
    df_final = pd.get_dummies(df_clean, drop_first=True, dtype=float)

    return df_final

X_encoded_list_final = [preprocess_features_robust(X) for X in X_reduced_list]
X_encoded_list_final = [clean_names(X) for X in X_encoded_list_final]

In [11]:
from IPython.display import display, Markdown

# 1. DIAGNOSTIC: Check exact string values
display(Markdown("### --- Diagnostic Check ---"))
sample_df = X_encoded_list_final[0]

if 'tenure_status_household' in sample_df.columns:
    display(Markdown("**Unique values in 'tenure_status_household':**"))
    display(Markdown(str(sample_df['tenure_status_household'].unique())))
else:
    display(Markdown("❌ 'tenure_status_household' is missing entirely from input data!"))

if 'urbanicity_cat' in sample_df.columns:
    display(Markdown("**Unique values in 'urbanicity_cat':**"))
    display(Markdown(str(sample_df['urbanicity_cat'].unique())))

### --- Diagnostic Check ---

**Unique values in 'tenure_status_household':**

[3 0 1 2 4]

**Unique values in 'urbanicity_cat':**

[0 1 2]

We recoded first substance use so small categories are grouped into Others

In [12]:
# Columns to combine
cols_to_group = [
    "first_sub_used_opioids",
    "first_sub_used_others",
    "first_sub_used_hallucinogens",
    "first_sub_used_inhalants",
    "first_sub_used_tranquilizers_hypnotics",
    "first_sub_used_amphetamine_type_stimulants",
]

# Loop over datasets 0–4 and modify in place
for i in range(5):
    df = X_encoded_list_final[i].copy()
    # Collapse into one dummy: if any of these == 1, mark as 1
    df["first_sub_used_other"] = df[cols_to_group].max(axis=1)
    # Drop the rest except the new combined column
    df = df.drop(columns=[c for c in cols_to_group if c != "first_sub_used_other"])
    # Replace the dataset in the original list
    X_encoded_list_final[i] = df

In [13]:
import sys
fold_output(
    "Show first db of X_encoded_list_final (newer) glimpse",
    lambda: glimpse(X_encoded_list_final[0])
)

For each imputed dataset, we fitted two regularized Cox models (one for readmission and one for death) using Coxnet, which applies elastic-net penalization with a strong LASSO component to enable variable selection. The loop fits both models on every imputation, prints basic model information, and stores all fitted models so they can later be combined or compared across imputations.

### Create bins for followup (landmarks)

We extracted the observed event times and corresponding event indicators directly from the structured survival objects (`y_surv_readm` and `y_surv_death`). Using the observed event times, we constructed evaluation grids based on the 5th to 95th percentiles of the event-time distribution. These grids define standardized time points at which model performance is assessed for both readmission and mortality outcomes.


In [14]:
import numpy as np
from IPython.display import display, Markdown

# Extract event times directly from structured arrays
event_times_readm = y_surv_readm["time"][y_surv_readm["event"]]
event_times_death = y_surv_death["time"][y_surv_death["event"]]

# Build evaluation grids (5th–95th percentiles, 50 points)
times_eval_readm = np.unique(
    np.quantile(event_times_readm, np.linspace(0.05, 0.95, 50))
)

times_eval_death = np.unique(
    np.quantile(event_times_death, np.linspace(0.05, 0.95, 50))
)

# Display only final result
display(Markdown(
    f"**Eval times (readmission):** `{times_eval_readm[:5]}` ... `{times_eval_readm[-5:]}`"
))

display(Markdown(
    f"**Eval times (death):** `{times_eval_death[:5]}` ... `{times_eval_death[-5:]}`"
))

**Eval times (readmission):** `[0.38709677 0.67741935 1.03225806 1.41935484 1.76666667]` ... `[46.81833443 50.96030612 55.16129032 60.84848585 67.08322581]`

**Eval times (death):** `[0.         0.09677419 1.06666667 2.1691691  3.34812377]` ... `[74.72632653 78.4516129  82.39472203 86.41935484 92.36311828]`

### Correct inmortal time bias

First, we eliminated inmortal time bias (dead patients look like without readmission).

This correction is essentially the Cause-Specific Hazard preparation. It is the correct way to handle Aim 3 unless you switch to a Fine-Gray model (which treats death as a specific type of event 2, rather than censoring 0). For RSF/Coxnet, censoring 0 is the correct approach.

In [15]:
import numpy as np

# Step 3. Replicate across imputations (safe copies)
n_imputations = len(X_encoded_list_final)
y_surv_readm_list = [y_surv_readm.copy() for _ in range(n_imputations)]
y_surv_death_list = [y_surv_death.copy() for _ in range(n_imputations)]

def correct_competing_risks(y_readm_list, y_death_list):
    corrected = []
    for y_readm, y_death in zip(y_readm_list, y_death_list):
        y_corr = y_readm.copy()

        # death observed and occurs before (or at) readmission/censoring time
        mask = (y_death["event"]) & (y_death["time"] < y_corr["time"])

        y_corr["event"][mask] = False
        y_corr["time"][mask] = y_death["time"][mask]

        corrected.append(y_corr)
    return corrected

# Step 4. Apply correction
y_surv_readm_list_corrected = correct_competing_risks(
    y_surv_readm_list,
    y_surv_death_list
)

In [16]:
# Check type and length
type(y_surv_readm_list_corrected), len(y_surv_readm_list_corrected)

# Look at the first element
y_surv_readm_list_corrected[0][:5]   # first 5 rows

array([(False, 68.96774194), ( True,  7.        ), ( True, 13.25806452),
       ( True,  5.        ), ( True,  7.35483871)],
      dtype=[('event', '?'), ('time', '<f8')])

In [17]:
from IPython.display import display, HTML
import html

def nb_print(*args, sep=" "):
    msg = sep.join(str(a) for a in args)
    display(HTML(f"<pre style='margin:0'>{html.escape(msg)}</pre>"))

The fully preprocessed and encoded feature matrices were renamed from `X_encoded_list_final` to `imputations_list_feb26` to reflect the finalized February 2026 analytic version of the imputed datasets. 

This object contains the harmonized, ordinal-encoded, and one-hot encoded predictor matrices for all five imputations and will serve as the definitive input for subsequent modeling procedures.

In [18]:
imputations_list_feb26 = X_encoded_list_final
del X_encoded_list_final

## Train / test split (80/20)

1. **Sets a fixed random seed** to make the 80/20 split exactly reproducible.

2. **Verifies required datasets exist** (features and survival outcomes) before doing anything.

3. **Creates a “death-corrected” outcome list** if it was not already available.

4. **Derives stratification labels** from treatment plan and completion categories plus readmission/death events.

5. **Uses a step-down strategy** if some strata are too rare, simplifying the stratification to keep it feasible.

6. **Caches a “full snapshot”** of all imputations and outcomes so reruns don’t silently change the split.

7. **Checks row alignment** so every imputation and every outcome has the same number of observations.

8. **Optionally checks stability across imputations** for plan/completion columns (should not vary much).

9. **Loads split indices from disk when available**, ensuring the exact same train/test split across sessions.

10. **Builds train/test datasets consistently for all imputations**, then runs strict diagnostics to confirm balance.


In [19]:
#@title 🧪 / 🎓 Reproducible 80/20 split before ML (integrated + idempotent + persisted)
# Stratification hierarchy:
#   1) plan + completion + readm_event + death_event
#   2) mixed fallback for rare full strata (<2) -> plan + readm + death
#   3) full fallback -> plan + readm + death

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from IPython.display import display, Markdown

SEED = 2125
TEST_SIZE = 0.20
FORCE_RESPLIT = False          # True to force new split
STRICT_SPLIT_CHECKS = True     # CI-style hard checks
MAX_EVENT_GAP = 0.01           # 1% tolerance
PERSIST_SPLIT_INDICES = True

SPLIT_FILE = Path("_out") / f"readm_split_seed{SEED}_test{int(TEST_SIZE*100)}.npz"
SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)

def nb_print_md(msg):
    display(Markdown(str(msg)))

# ---------- Requirements ----------
required = [
    "imputations_list_feb26",
    "y_surv_readm_list",
    "y_surv_readm_list_corrected",
    "y_surv_death_list",
]
missing = [v for v in required if v not in globals()]
if missing:
    raise ValueError(f"Missing required objects: {missing}")

if "y_surv_death_list_corrected" not in globals():
    y_surv_death_list_corrected = [y.copy() for y in y_surv_death_list]

# ---------- Helpers ----------
def get_plan_labels(df):
    labels = np.zeros(len(df), dtype=int)  # 0 = reference
    if "plan_type_corr_pg_pr" in df.columns:
        labels[pd.to_numeric(df["plan_type_corr_pg_pr"], errors="coerce").fillna(0).to_numpy() == 1] = 1
    if "plan_type_corr_m_pr" in df.columns:
        labels[pd.to_numeric(df["plan_type_corr_m_pr"], errors="coerce").fillna(0).to_numpy() == 1] = 2
    if "plan_type_corr_pg_pai" in df.columns:
        labels[pd.to_numeric(df["plan_type_corr_pg_pai"], errors="coerce").fillna(0).to_numpy() == 1] = 3
    if "plan_type_corr_m_pai" in df.columns:
        labels[pd.to_numeric(df["plan_type_corr_m_pai"], errors="coerce").fillna(0).to_numpy() == 1] = 4
    return labels

def get_completion_labels(df):
    labels = np.zeros(len(df), dtype=int)  # 0 = completion/reference (all listed dummies = 0)
    if "tr_outcome_referral" in df.columns:
        labels[pd.to_numeric(df["tr_outcome_referral"], errors="coerce").fillna(0).to_numpy() == 1] = 1
    if "tr_outcome_dropout" in df.columns:
        labels[pd.to_numeric(df["tr_outcome_dropout"], errors="coerce").fillna(0).to_numpy() == 1] = 2
    if "tr_outcome_adm_discharge_rule_violation_undet" in df.columns:
        labels[pd.to_numeric(df["tr_outcome_adm_discharge_rule_violation_undet"], errors="coerce").fillna(0).to_numpy() == 1] = 3
    if "tr_outcome_adm_discharge_adm_reasons" in df.columns:
        labels[pd.to_numeric(df["tr_outcome_adm_discharge_adm_reasons"], errors="coerce").fillna(0).to_numpy() == 1] = 4
    if "tr_outcome_other" in df.columns:
        labels[pd.to_numeric(df["tr_outcome_other"], errors="coerce").fillna(0).to_numpy() == 1] = 5
    return labels

def build_strata(X0, y_readm0, y_death0):
    """
    Build stratification labels with progressive fallback:

    1) full: plan + completion + readmission_event + death_event
    2) mixed: only rare full strata (<2 rows) are replaced by fallback labels
       (plan + readmission_event + death_event)
    3) fallback: plan + readmission_event + death_event for all rows

    Returns:
        strata (np.ndarray), mode (str), readm_evt (np.ndarray), death_evt (np.ndarray)
    """
    plan = get_plan_labels(X0)
    comp = get_completion_labels(X0)
    readm_evt = y_readm0["event"].astype(int)
    death_evt = y_death0["event"].astype(int)

    full = pd.Series(plan.astype(str) + "_" + comp.astype(str) + "_" + readm_evt.astype(str) + "_" + death_evt.astype(str))
    if full.value_counts().min() >= 2:
        return full.to_numpy(), "full(plan+completion+readm+death)", readm_evt, death_evt

    fb = pd.Series(plan.astype(str) + "_" + readm_evt.astype(str) + "_" + death_evt.astype(str))
    mixed = full.copy()
    rare = mixed.map(mixed.value_counts()) < 2
    mixed[rare] = fb[rare]  # merge rare rows into existing fallback strata
    if mixed.value_counts().min() >= 2:
        return mixed.to_numpy(), "mixed(rare->plan+readm+death)", readm_evt, death_evt

    if fb.value_counts().min() >= 2:
        return fb.to_numpy(), "fallback(plan+readm+death)", readm_evt, death_evt

    raise ValueError("Could not build stratification labels with >=2 rows per stratum.")

def split_df_list(df_list, tr_idx, te_idx):
    tr = [df.iloc[tr_idx].reset_index(drop=True).copy() for df in df_list]
    te = [df.iloc[te_idx].reset_index(drop=True).copy() for df in df_list]
    return tr, te

def split_surv_list(y_list, tr_idx, te_idx):
    tr = [y[tr_idx].copy() for y in y_list]
    te = [y[te_idx].copy() for y in y_list]
    return tr, te

# ---------- Cache full data once (idempotent re-runs) ----------
if "_split_cache_readm_feb26" not in globals():
    _split_cache_readm_feb26 = {}
cache = _split_cache_readm_feb26

if FORCE_RESPLIT:
    cache.pop("idx", None)

if FORCE_RESPLIT or "full" not in cache:
    cache["full"] = {
        "X": [df.reset_index(drop=True).copy() for df in imputations_list_feb26],
        "y_readm": [y.copy() for y in y_surv_readm_list],
        "y_readm_corr": [y.copy() for y in y_surv_readm_list_corrected],
        "y_death": [y.copy() for y in y_surv_death_list],
        "y_death_corr": [y.copy() for y in y_surv_death_list_corrected],
    }

full = cache["full"]

# ---------- Consistency checks ----------
n_imp = len(full["X"])
n = len(full["X"][0])

if any(len(df) != n for df in full["X"]):
    raise ValueError("Row mismatch inside full X list.")

for name, obj in [
    ("y_readm", full["y_readm"]),
    ("y_readm_corr", full["y_readm_corr"]),
    ("y_death", full["y_death"]),
    ("y_death_corr", full["y_death_corr"]),
]:
    if len(obj) != n_imp:
        raise ValueError(f"{name} length ({len(obj)}) != n_imputations ({n_imp})")
    if any(len(y) != n for y in obj):
        raise ValueError(f"Row mismatch between X and {name}.")

# ---------- Optional diagnostic: plan/completion consistency across imputations ----------
plan_comp_cols = [
    c for c in [
        "plan_type_corr_pg_pr",
        "plan_type_corr_m_pr",
        "plan_type_corr_pg_pai",
        "plan_type_corr_m_pai",
        "tr_outcome_referral",
        "tr_outcome_dropout",
        "tr_outcome_adm_discharge_rule_violation_undet",
        "tr_outcome_adm_discharge_adm_reasons",
        "tr_outcome_other",
    ] if c in full["X"][0].columns
]

max_diff_rows = 0
if plan_comp_cols:
    base_pc = full["X"][0][plan_comp_cols].astype("string").fillna("__NA__").reset_index(drop=True)
    for i in range(1, n_imp):
        cur_pc = full["X"][i][plan_comp_cols].astype("string").fillna("__NA__").reset_index(drop=True)
        diff_rows = int((base_pc != cur_pc).any(axis=1).sum())
        max_diff_rows = max(max_diff_rows, diff_rows)

# ---------- Try loading indices from disk ----------
loaded_from_disk = False
if PERSIST_SPLIT_INDICES and (not FORCE_RESPLIT) and SPLIT_FILE.exists() and ("idx" not in cache):
    z = np.load(SPLIT_FILE, allow_pickle=False)
    tr = z["train_idx"].astype(int)
    te = z["test_idx"].astype(int)
    n_disk = int(z["n_full"][0]) if "n_full" in z else n
    if n_disk == n and tr.max() < n and te.max() < n:
        cache["idx"] = (np.sort(tr), np.sort(te))
        cache["strat_mode"] = str(z["strat_mode"][0]) if "strat_mode" in z else "loaded_from_disk"
        loaded_from_disk = True

# ---------- Compute or reuse split indices ----------
if FORCE_RESPLIT or "idx" not in cache:
    strata_used, strat_mode, readm_evt_all, death_evt_all = build_strata(
        full["X"][0], full["y_readm"][0], full["y_death"][0]
    )
    idx = np.arange(n)
    train_idx, test_idx = train_test_split(
        idx, test_size=TEST_SIZE, random_state=SEED, shuffle=True, stratify=strata_used
    )
    train_idx = np.sort(train_idx)
    test_idx = np.sort(test_idx)
    cache["idx"] = (train_idx, test_idx)
    cache["strat_mode"] = strat_mode

    if PERSIST_SPLIT_INDICES:
        np.savez_compressed(
            SPLIT_FILE,
            train_idx=train_idx,
            test_idx=test_idx,
            n_full=np.array([n], dtype=int),
            seed=np.array([SEED], dtype=int),
            test_size=np.array([TEST_SIZE], dtype=float),
            strat_mode=np.array([strat_mode], dtype="U64"),
        )
else:
    train_idx, test_idx = cache["idx"]
    train_idx = np.sort(train_idx)
    test_idx = np.sort(test_idx)
    readm_evt_all = full["y_readm"][0]["event"].astype(int)
    death_evt_all = full["y_death"][0]["event"].astype(int)

# ---------- Build train/test from full snapshot every run ----------
imputations_list_feb26_train, imputations_list_feb26_test = split_df_list(full["X"], train_idx, test_idx)

y_surv_readm_list_train, y_surv_readm_list_test = split_surv_list(full["y_readm"], train_idx, test_idx)
y_surv_readm_list_corrected_train, y_surv_readm_list_corrected_test = split_surv_list(full["y_readm_corr"], train_idx, test_idx)

y_surv_death_list_train, y_surv_death_list_test = split_surv_list(full["y_death"], train_idx, test_idx)
y_surv_death_list_corrected_train, y_surv_death_list_corrected_test = split_surv_list(full["y_death_corr"], train_idx, test_idx)

# Downstream code uses TRAIN only
imputations_list_feb26 = imputations_list_feb26_train
y_surv_readm_list = y_surv_readm_list_train
y_surv_readm_list_corrected = y_surv_readm_list_corrected_train
y_surv_death_list = y_surv_death_list_train
y_surv_death_list_corrected = y_surv_death_list_corrected_train

# ---------- Diagnostics + strict checks ----------
strata_diag, strat_mode_diag, _, _ = build_strata(full["X"][0], full["y_readm"][0], full["y_death"][0])
sdiag = pd.Series(strata_diag)
train_strata = set(sdiag.iloc[train_idx].unique())
test_strata = set(sdiag.iloc[test_idx].unique())
missing_in_test = sorted(train_strata - test_strata)
missing_in_train = sorted(test_strata - train_strata)

readm_gap = abs(readm_evt_all[train_idx].mean() - readm_evt_all[test_idx].mean())
death_gap = abs(death_evt_all[train_idx].mean() - death_evt_all[test_idx].mean())

# full-strata rarity report (before fallback)
strata_full = pd.Series(
    get_plan_labels(full["X"][0]).astype(str) + "_" +
    get_completion_labels(full["X"][0]).astype(str) + "_" +
    full["y_readm"][0]["event"].astype(int).astype(str) + "_" +
    full["y_death"][0]["event"].astype(int).astype(str)
)
vc_full = strata_full.value_counts()
rare_rows = int((strata_full.map(vc_full) < 2).sum())

if STRICT_SPLIT_CHECKS:
    assert len(np.intersect1d(train_idx, test_idx)) == 0, "Train/Test index overlap detected."
    assert (len(train_idx) + len(test_idx)) == n, "Train/Test sizes do not sum to n."
    assert len(missing_in_test) == 0, f"Strata missing in test: {missing_in_test}"
    assert len(missing_in_train) == 0, f"Strata missing in train: {missing_in_train}"
    assert readm_gap < MAX_EVENT_GAP, f"Readmission rate imbalance > {MAX_EVENT_GAP:.0%} (gap={readm_gap:.4f})"
    assert death_gap < MAX_EVENT_GAP, f"Death rate imbalance > {MAX_EVENT_GAP:.0%} (gap={death_gap:.4f})"

# ---------- Summary ----------
nb_print_md(f"**Loaded indices from disk:** `{loaded_from_disk}`")
nb_print_md(f"**Split file:** `{SPLIT_FILE}`")
nb_print_md(f"**Split mode used:** `{cache.get('strat_mode', strat_mode_diag)}`")
nb_print_md(f"**Plan/completion diff rows across imputations (max vs imp0):** `{max_diff_rows}`")
nb_print_md(f"**Full strata count:** `{vc_full.size}` | **Min full stratum size:** `{int(vc_full.min())}` | **Rows in rare full strata (<2):** `{rare_rows}`")
nb_print_md(f"**Train/Test sizes:** `{len(train_idx)}` ({len(train_idx)/n:.1%}) / `{len(test_idx)}` ({len(test_idx)/n:.1%})")
nb_print_md(
    "**Readmission rate all/train/test:** "
    f"`{readm_evt_all.mean():.3%}` / `{readm_evt_all[train_idx].mean():.3%}` / `{readm_evt_all[test_idx].mean():.3%}`"
)
nb_print_md(
    "**Death rate all/train/test:** "
    f"`{death_evt_all.mean():.3%}` / `{death_evt_all[train_idx].mean():.3%}` / `{death_evt_all[test_idx].mean():.3%}`"
)
nb_print_md(
    f"**Strata in train/test:** `{len(train_strata)}` / `{len(test_strata)}` | "
    f"**Missing train→test:** `{len(missing_in_test)}` | **Missing test→train:** `{len(missing_in_train)}`"
)


**Loaded indices from disk:** `True`

**Split file:** `_out\readm_split_seed2125_test20.npz`

**Split mode used:** `fallback(plan+readm+death)`

**Plan/completion diff rows across imputations (max vs imp0):** `0`

**Full strata count:** `107` | **Min full stratum size:** `1` | **Rows in rare full strata (<2):** `4`

**Train/Test sizes:** `70803` (80.0%) / `17701` (20.0%)

**Readmission rate all/train/test:** `21.547%` / `21.547%` / `21.547%`

**Death rate all/train/test:** `4.460%` / `4.459%` / `4.463%`

**Strata in train/test:** `20` / `20` | **Missing train→test:** `0` | **Missing test→train:** `0`

In [20]:
# counts per stratum in train/test
train_counts = sdiag.iloc[train_idx].value_counts()
test_counts  = sdiag.iloc[test_idx].value_counts()

min_train = int(train_counts.min())
min_test  = int(test_counts.min())

nb_print_md(f"**Min stratum count in TRAIN (used strata):** `{min_train}`")
nb_print_md(f"**Min stratum count in TEST (used strata):** `{min_test}`")

# strata that got 0 in test or 0 in train
zero_in_test = sorted(set(train_counts.index) - set(test_counts.index))
zero_in_train = sorted(set(test_counts.index) - set(train_counts.index))

nb_print_md(f"**Strata with 0 in TEST:** `{len(zero_in_test)}`")
nb_print_md(f"**Strata with 0 in TRAIN:** `{len(zero_in_train)}`")

# show examples with their full-data counts
if len(zero_in_test) > 0:
    ex = zero_in_test[:10]
    nb_print_md(f"**Examples 0 in TEST (up to 10):** `{ex}`")
    nb_print_md(f"**Full-data counts:** `{[int(sdiag.value_counts()[k]) for k in ex]}`")

**Min stratum count in TRAIN (used strata):** `18`

**Min stratum count in TEST (used strata):** `5`

**Strata with 0 in TEST:** `0`

**Strata with 0 in TRAIN:** `0`

In [21]:
strata_full = pd.Series(
    get_plan_labels(full["X"][0]).astype(str) + "_" +
    get_completion_labels(full["X"][0]).astype(str) + "_" +
    full["y_readm"][0]["event"].astype(int).astype(str) + "_" +
    full["y_death"][0]["event"].astype(int).astype(str)
)

vc = strata_full.value_counts()
display(Markdown(f"**# full strata:** `{vc.size}`"))
display(Markdown(f"**Min stratum size (full):** `{int(vc.min())}`"))
display(Markdown(f"**# strata with count < 2:** `{int((vc < 2).sum())}`"))

**# full strata:** `107`

**Min stratum size (full):** `1`

**# strata with count < 2:** `4`

In [22]:
plan = get_plan_labels(full["X"][0])
readm_evt = full["y_readm"][0]["event"].astype(int)
death_evt = full["y_death"][0]["event"].astype(int)

fb = pd.Series(plan.astype(str) + "_" + readm_evt.astype(str) + "_" + death_evt.astype(str))

rare_mask = strata_full.map(strata_full.value_counts()) < 2
n_rare = int(rare_mask.sum())

display(Markdown(f"**Rows in rare full-strata (<2):** `{n_rare}`"))
if n_rare > 0:
    display(Markdown(
        f"**Rare rows proportion:** `{n_rare/len(strata_full):.3%}`"
    ))

**Rows in rare full-strata (<2):** `4`

**Rare rows proportion:** `0.005%`

In [23]:
# Use the actual stratification mode that was used to split
strata_used, strat_mode, _, _ = build_strata(full["X"][0], full["y_readm"][0], full["y_death"][0])

s = pd.Series(strata_used)
train_strata = set(s.iloc[train_idx].unique())
test_strata = set(s.iloc[test_idx].unique())

missing_in_test = sorted(train_strata - test_strata)
missing_in_train = sorted(test_strata - train_strata)

display(Markdown(f"**Strata used:** `{strat_mode}`"))
display(Markdown(f"**# strata in train:** `{len(train_strata)}` | **# strata in test:** `{len(test_strata)}`"))
display(Markdown(f"**Strata present in train but missing in test:** `{len(missing_in_test)}`"))
display(Markdown(f"**Strata present in test but missing in train:** `{len(missing_in_train)}`"))

**Strata used:** `fallback(plan+readm+death)`

**# strata in train:** `20` | **# strata in test:** `20`

**Strata present in train but missing in test:** `0`

**Strata present in test but missing in train:** `0`

In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Absolute directory
OUT_DIR = Path(r"G:\My Drive\Alvacast\SISTRAT 2023\cons\_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PARQUET = OUT_DIR / f"death_split_seed{SEED}_test{int(TEST_SIZE*100)}.parquet"

split_df = pd.DataFrame({
    "row_id": np.arange(n),
    "is_train": np.isin(np.arange(n), train_idx)
})

split_df.to_parquet(SPLIT_PARQUET, index=False)

display(Markdown(f"**Saved split to:** `{SPLIT_PARQUET}`"))

**Saved split to:** `G:\My Drive\Alvacast\SISTRAT 2023\cons\_out\death_split_seed2125_test20.parquet`

We cleaned our environment safely so that:

- Old models

- Temporary objects

- Large intermediate datasets

do not interfere with the next modeling block.

In [25]:
# Safe cleanup before Readmission XGBoost blocks
import types
import gc

# Ensure logger exists (some target cells expect it)
if "nb_print" not in globals():
    def nb_print(*args, **kwargs):
        print(*args, **kwargs)

# Compatibility: one Optuna/Bootstrap block checks jan26 naming
#if "imputations_list_jan26" not in globals() and "imputations_list_feb26" in globals():
#    imputations_list_jan26 = imputations_list_feb26

KEEP = {
    "nb_print", "study",
    "imputations_list_feb26", #"imputations_list",#"imputations_list_jan26"
    "X_train", "y_surv_readm_list_corrected", "y_surv_readm_list", "y_surv_death_list",
    # Optional plot config objects:
    "plt", "sns", "matplotlib", "mpl", "rcParams",
}

for name, obj in list(globals().items()):
    if name in KEEP or name.startswith("_"):
        continue
    if isinstance(obj, types.ModuleType):   # keep imports
        continue
    if callable(obj):                        # keep functions/classes
        continue
    del globals()[name]

gc.collect()

required = ["y_surv_readm_list_corrected", "y_surv_readm_list", "y_surv_death_list"]
missing = [x for x in required if x not in globals()]
print("Missing required objects:", missing)


## ML

### Advanced Survival Modeling: XGBoost & Stratified Evaluation

In this section, we transition to a Gradient Boosted Decision Tree (GBDT) framework using XGBoost. This serves as a robust non-linear benchmark to complement the neural network analysis for low-event survival data (approximately 4% death events).

#### Methodological Framework
* **Cox-Objective Boosting:** We use the `survival:cox` objective, which optimizes the Cox partial log-likelihood within a boosting architecture. This enables flexible non-linear risk modeling and interaction learning while retaining the proportional hazards formulation.

* **5-Fold Cross-Validation with Stratification (Death Model):** We use 5-fold cross-validation with stratification to preserve key data structure across folds. In the death model, stratification is based on a **combined label of treatment plan type and event status**, with fallback to simpler stratification when rare strata make 5-fold splitting infeasible. (The readmission model currently uses plan-type-only stratification.)

* **Censoring-Aware Evaluation:** Hyperparameter selection is based on **Uno’s C-Index (IPCW)**, which is appropriate under right censoring. We also report the **Integrated Brier Score (IBS)** as a complementary calibration/overall prediction error metric.

#### Hyperparameter Optimization
Given the low event rate, we run a randomized search over a dense parameter grid. The search emphasizes regularization and tree-complexity controls (`min_child_weight`, `gamma`, `reg_alpha`, `reg_lambda`) to improve generalization and reduce overfitting.

#### Breslow Estimation
Because XGBoost with `survival:cox` outputs relative risk scores, we estimate the baseline hazard/survival using the **Breslow estimator** to derive absolute survival probabilities 𝑆(𝑡∣𝑥), enabling time-specific calibration metrics such as IBS.


### Parameter tuning

In [ ]:
#@title ⚡ XGBoost Death Robust Tuning (100 Iterations, CPU Only, Dual Stratification + Fallback)
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, KFold, ParameterSampler
from sksurv.metrics import concordance_index_ipcw
import time
import gc
import os
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

# Fallback in case nb_print is not defined globally
if 'nb_print' not in globals():
    def nb_print(*args, **kwargs):
        print(*args, **kwargs)

total_start_time = time.time()

# --- CPU CONFIGURATION ---
N_CORES = max(1, os.cpu_count() - 2)
nb_print(f"Parallel Execution Configured: Using {N_CORES} CPU cores.")

# --- 1. SETUP & DATA ---
nb_print("Preparing data for Robust XGBoost Tuning (Death)...")

try:
    if 'imputations_list_feb26' in locals():
        df_tune = imputations_list_feb26[0].copy()
        y_tune_struct = y_surv_death_list[0]
    else:
        df_tune = X_train.copy()
        y_tune_struct = y_surv_death_list[0]

    nb_print(f"  Data Shape: {df_tune.shape}")
    nb_print(f"  Target: Death (Events: {np.asarray(y_tune_struct['event']).sum()})")

except Exception as e:
    raise ValueError(f"Data Error: {e}. Please run data loading steps first.")

# --- 2. STRATIFICATION HELPERS ---
def get_plan_stratification_labels(df):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    return labels

def get_dual_stratification_labels(df, y_struct):
    labels = get_plan_stratification_labels(df)
    event_status = np.asarray(y_struct['event']).astype(int)
    return (labels * 10) + event_status

def choose_stratification(df, y_struct, n_splits=5):
    dual_labels = get_dual_stratification_labels(df, y_struct)
    dual_u, dual_c = np.unique(dual_labels, return_counts=True)

    if len(dual_u) > 1 and dual_c.min() >= n_splits:
        nb_print("Stratification mode: dual (plan_type x event).")
        return dual_labels, "dual", True

    rare_dual = {int(k): int(v) for k, v in zip(dual_u, dual_c) if v < n_splits}
    nb_print(f"[Fallback triggered] Dual stratification has classes with < {n_splits} samples: {rare_dual}")

    plan_labels = get_plan_stratification_labels(df)
    plan_u, plan_c = np.unique(plan_labels, return_counts=True)
    if len(plan_u) > 1 and plan_c.min() >= n_splits:
        nb_print("Stratification mode: plan_type only (fallback).")
        return plan_labels, "plan_only", True

    event_labels = np.asarray(y_struct['event']).astype(int)
    ev_u, ev_c = np.unique(event_labels, return_counts=True)
    if len(ev_u) > 1 and ev_c.min() >= n_splits:
        nb_print("Stratification mode: event only (fallback).")
        return event_labels, "event_only", True

    nb_print(f"[Fallback triggered] No valid stratification for {n_splits} folds. Using unstratified KFold.")
    return None, "kfold", False

strat_labels, strat_mode, use_stratified = choose_stratification(df_tune, y_tune_struct, n_splits=5)
y_xgb_label = np.where(np.asarray(y_tune_struct['event']), np.asarray(y_tune_struct['time']), -np.asarray(y_tune_struct['time']))

# --- 3. SEARCH SPACE (ORIGINAL DEATH PARAM RANGES) ---
param_grid = {
    'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 5, 10, 20, 50],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 1, 5, 10],
    'reg_lambda': [0.1, 1, 5, 10, 20],
    'gamma': [0, 0.1, 0.5, 1, 2]
}

N_ITER = 100
param_list = list(ParameterSampler(param_grid, n_iter=N_ITER, random_state=2125))  # 42 -> 2125

# --- 4. TUNING LOOP ---
nb_print(f"\nStarting Exhaustive Search ({N_ITER} combos)...")
nb_print(f"  Strategy: 5-Fold CV | Stratification mode: {strat_mode}")
nb_print("  Metric: Uno's C-Index (IPCW)")

results = []
if use_stratified:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
else:
    cv = KFold(n_splits=5, shuffle=True, random_state=2125)

for i, sampled_params in enumerate(param_list, start=1):
    params = sampled_params.copy()
    params['objective'] = 'survival:cox'
    params['eval_metric'] = 'cox-nloglik'
    params['tree_method'] = 'hist'
    params['seed'] = 2125
    params['nthread'] = N_CORES
    params['device'] = 'cpu'
    params['verbosity'] = 0

    fold_scores = []

    if use_stratified:
        split_iter = cv.split(df_tune, strat_labels)
    else:
        split_iter = cv.split(df_tune)

    for train_idx, val_idx in split_iter:
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_xgb, y_va_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        y_tr_struct, y_va_struct = y_tune_struct[train_idx], y_tune_struct[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_xgb)

        model = xgb.train(
            params, dtrain, num_boost_round=1500,
            evals=[(dval, 'val')], early_stopping_rounds=30,
            verbose_eval=False
        )

        risk_scores = model.predict(dval)

        try:
            c_val = concordance_index_ipcw(y_tr_struct, y_va_struct, risk_scores)[0]
        except Exception:
            from sksurv.metrics import concordance_index_censored
            c_val = concordance_index_censored(
                np.asarray(y_va_struct['event']),
                np.asarray(y_va_struct['time']),
                risk_scores
            )[0]

        fold_scores.append(c_val)

        del model, dtrain, dval, risk_scores
        gc.collect()

    avg_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    results.append({**params, 'Unos_C_Index': avg_score, 'Std_Dev': std_score, 'Strat_Mode': strat_mode})

    if i % 5 == 0:
        elapsed_min = (time.time() - total_start_time) / 60
        best_so_far = max(r['Unos_C_Index'] for r in results)
        nb_print(f"  [{i}/{N_ITER}] Best: {best_so_far:.4f} | Current: {avg_score:.4f} | Elapsed: {elapsed_min:.2f} min")

# --- 5. FINALIZE & EXPORT ---
total_duration_min = (time.time() - total_start_time) / 60
nb_print(f"\nTotal Execution Time: {total_duration_min:.2f} minutes")

df_results = pd.DataFrame(results).sort_values(by='Unos_C_Index', ascending=False)
best_config = df_results.iloc[0].to_dict()

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"_out/XGB_Death_Robust_Tuning_5Fold_{timestamp_str}.csv"

os.makedirs("_out", exist_ok=True)
df_results.to_csv(filename, index=False)

nb_print("\nTuning Complete!")
nb_print(f"  Best C-Index: {best_config['Unos_C_Index']:.4f}")
nb_print(f"  Stratification used: {best_config['Strat_Mode']}")
nb_print(f"Saved to: {filename}")


~11 minutes

In [ ]:
nb_print(f"\nTuning Complete!")
nb_print(f"  Best C-Index: {best_config['Unos_C_Index']:.4f}")

In [ ]:
nb_print(best_config)

In [ ]:
import pandas as pd
from IPython.display import HTML, display

# Reset options so Pandas doesn't force everything
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Convert DataFrame to HTML and wrap in a scrollable div
html_table = df_results.to_html()
scroll_box = f"""
<div style="max-height:500px; max-width:1000px; overflow-y:auto; overflow-x:auto; border:1px solid #ccc;">
{html_table}
</div>
"""
display(HTML(scroll_box))


,subsample,reg_lambda,reg_alpha,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree,objective,eval_metric,tree_method,seed,nthread,device,verbosity,Unos_C_Index,Std_Dev
16,0.9,0.1,0.1,1,5,0.010,0.1,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.745007,0.017049
41,0.9,1.0,1.0,10,3,0.010,0.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744481,0.016503
44,0.7,1.0,0.1,10,4,0.010,0.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744270,0.015859
32,0.6,1.0,0.0,1,4,0.010,0.1,0.8,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744245,0.015812
20,0.7,5.0,5.0,10,4,0.005,0.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744180,0.015198
43,0.8,0.1,0.1,10,3,0.010,0.0,0.8,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743670,0.015671
12,0.9,1.0,0.1,10,3,0.005,2.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743611,0.016138
17,0.8,5.0,10.0,1,5,0.010,1.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743482,0.015742
7,0.8,1.0,10.0,20,8,0.010,2.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743223,0.016286
19,0.6,0.1,0.1,20,8,0.005,0.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743158,0.015927


### Optuna

- Multi-objective tuning balances C-index and IBS.
- Uses 5-fold stratified cross-validation.
- Evaluates performance at 5 clinical time horizons.
- Averages time-specific C-indices for robustness.
- Computes survival via Breslow baseline hazard.
- Converts risk scores into survival probabilities.
- Uses IPCW C-index for censoring adjustment.
- Applies early pruning for poor-performing trials.
- Returns Pareto-optimal models, not a single winner.


In [ ]:
# @title Optuna Multi-Objective: C-Index (Discrimination) vs IBS (Calibration)
import optuna
import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import os
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw, brier_score

# --- 1. SETUP & CPU CONFIGURATION ---
N_CORES = max(1, os.cpu_count() - 2)
N_ITER = 100
EVAL_HORIZONS = [3, 6, 12, 36, 60]

# df_tune & y_tune_struct exist (imputations_list_jan26 already in memory)
df_tune = imputations_list_jan26[0].copy()
y_tune_struct = y_surv_death_list[0]
y_xgb_label = np.where(y_tune_struct['event'], y_tune_struct['time'], -y_tune_struct['time'])

# --- 2. DUAL STRATIFICATION HELPER ---
def get_dual_stratification_labels(df, y_struct):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    
    event_status = y_struct['event'].astype(int)
    return (labels * 10) + event_status

strat_labels_dual = get_dual_stratification_labels(df_tune, y_tune_struct)

# --- 3. HELPER: BRESLOW SURVIVAL ---
def predict_survival_probs_breslow(y_tr, risk_tr, risk_va, eval_times):
    if np.any(risk_tr <= 0):
        risk_tr = np.exp(risk_tr)
        risk_va = np.exp(risk_va)

    order = np.argsort(y_tr['time'])
    t_train = y_tr['time'][order]
    e_train = y_tr['event'][order]
    risk_train_ord = risk_tr[order]
    
    unique_times = np.unique(t_train[e_train])
    baseline_hazard = np.zeros_like(unique_times, dtype=float)
    
    for i, t in enumerate(unique_times):
        at_risk = t_train >= t
        events_at_t = np.sum((t_train == t) & e_train)
        baseline_hazard[i] = events_at_t / np.sum(risk_train_ord[at_risk])
        
    cum_baseline_hazard = np.cumsum(baseline_hazard)
    
    surv_probs = np.zeros((len(risk_va), len(eval_times)))
    for j, tau in enumerate(eval_times):
        valid_idx = np.where(unique_times <= tau)[0]
        H0_t = cum_baseline_hazard[valid_idx[-1]] if len(valid_idx) > 0 else 0.0
        surv_probs[:, j] = np.exp(-H0_t * risk_va) 
        
    return surv_probs

# --- 4. OPTUNA OBJECTIVE ---
def objective(trial):
    params = {
        'objective': 'survival:cox',
        'eval_metric': 'cox-nloglik',
        'tree_method': 'hist',
        'device': 'cpu',
        'nthread': N_CORES,  # <-- Added to parallel processing
        'verbosity': 0,
        'seed': 2125, 
        
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.02, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 30),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0)
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
    
    fold_c_indices = []
    fold_ib_scores = []
    fold_global_c_indices = [] 

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_tune, strat_labels_dual)):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_xgb, y_va_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        y_tr_struct, y_va_struct = y_tune_struct[train_idx], y_tune_struct[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_xgb)

        model = xgb.train(
            params, dtrain, 
            num_boost_round=1500,
            evals=[(dval, 'val')], 
            early_stopping_rounds=30, 
            verbose_eval=False
        )

        risk_tr = model.predict(dtrain)
        risk_va = model.predict(dval)
        
        # 1. MULTI-HORIZON C-INDEX
        h_c_indices = []
        for tau_val in EVAL_HORIZONS:
            try:
                c_val = concordance_index_ipcw(y_tr_struct, y_va_struct, risk_va, tau=tau_val)[0]
                h_c_indices.append(c_val)
            except:
                pass 
        avg_c_index = np.mean(h_c_indices) if len(h_c_indices) > 0 else 0.5
        
        # 2. GLOBAL C-INDEX 
        try:
            global_c = concordance_index_ipcw(y_tr_struct, y_va_struct, risk_va)[0]
        except:
            global_c = 0.5
        fold_global_c_indices.append(global_c)

        # 3. BRIER SCORE (IBS)
        try:
            surv_probs_va = predict_survival_probs_breslow(y_tr_struct, risk_tr, risk_va, EVAL_HORIZONS)
            _, brier_scores_at_tau = brier_score(y_tr_struct, y_va_struct, surv_probs_va, EVAL_HORIZONS)
            avg_ibs = np.mean(brier_scores_at_tau)
        except:
            avg_ibs = 0.25 

        fold_c_indices.append(avg_c_index)
        fold_ib_scores.append(avg_ibs)
            
        del model, dtrain, dval, risk_tr, risk_va
        gc.collect()

        # Pruning
        current_mean_c = np.mean(fold_c_indices)
        if fold_idx >= 1 and current_mean_c < 0.60:
            raise optuna.TrialPruned()

    trial.set_user_attr("Global_C_Index", np.mean(fold_global_c_indices))
    return np.mean(fold_c_indices), np.mean(fold_ib_scores)


# --- 5. MULTI-OBJECTIVE INITIALIZATION ---
study = optuna.create_study(
    directions=['maximize', 'minimize'], # <-- CORREGIDO PARA QUE SEAN DOS OBJETIVOS
    study_name="XGB_Death_Optuna_Fresh_Search"
)

print(f"\nInitiating search from start, ({N_ITER} combinaciones)...")
study.optimize(objective, n_trials=N_ITER, show_progress_bar=True)

# --- 6. EXTRACTION OF OPTIMAL MODELS ---
print("\nOptimal Models found (Pareto Front):")
best_trials = study.best_trials
for t in best_trials:
    global_c_val = t.user_attrs.get("Global_C_Index", "N/A")
    print(f"Trial {t.number} -> Multi-Horizon C: {t.values[0]:.4f} | IBS: {t.values[1]:.4f} | Global C: {global_c_val:.4f}")

[I 2026-02-23 10:30:53,636] A new study created in memory with name: XGB_Death_Optuna_Fresh_Search
100%|██████████| 50/50 [22:27<00:00, 26.95s/it]


~22 minutes

In [29]:
nb_print("\nOptimal Models found (Pareto Front):")
best_trials = study.best_trials
for t in best_trials:
    global_c_val = t.user_attrs.get("Global_C_Index", "N/A")
    nb_print(f"Trial {t.number} -> Multi-Horizon C: {t.values[0]:.4f} | IBS: {t.values[1]:.4f} | Global C: {global_c_val:.4f}")

In [18]:
import os
import joblib

# Make sure the folder exists before saving
os.makedirs("_input", exist_ok=True)

# --- 7. SAVE THE ENTIRE STUDY TO A .PKL FILE ---
study_filename = f"_input/XGB_Death_Optuna_Study_{timestamp_str}.pkl"

# Save study object
joblib.dump(study, study_filename)
print(f"Study object successfully saved to: {study_filename}")

# (Optional) Force download if running in Google Colab
try:
    from google.colab import files
    files.download(study_filename)
except Exception as e:
    print("Download skipped (not in Colab or browser blocked it).")


In [19]:
# @title Final Model Selection (Euclidean Distance to Ideal Point)
import pandas as pd
import numpy as np
from datetime import datetime

print("Analyzing the Pareto Front...")

# --- Load or reuse study object ---
import os
import joblib
import glob
import re
import datetime as _dt

# If an in-memory study with the specific study_name exists, use it.
# Otherwise, find the most recent saved study file matching the pattern and load it.
if 'study' in globals() and getattr(study, 'study_name', None) == "XGB_Death_Optuna_Fresh_Search":
    # Use the existing in-memory study; no file load required.
    print("Using in-memory study with study_name='XGB_Death_Optuna_Fresh_Search'.")
else:
    # Ensure the input folder exists before searching
    os.makedirs("_input", exist_ok=True)

    # Pattern for saved study files
    pattern = "_input/XGB_Death_Optuna_Study_*.pkl"
    files = glob.glob(pattern)

    if not files:
        raise FileNotFoundError("No saved study files found matching pattern: _input/XGB_Death_Optuna_Study_*.pkl")

    # Extract timestamp from filenames and pick the most recent one
    timestamped_files = []
    for f in files:
        m = re.search(r"XGB_Death_Optuna_Study_(\d{8}_\d{4})\.pkl$", f)
        if m:
            ts = m.group(1)
            try:
                dt = _dt.datetime.strptime(ts, "%Y%m%d_%H%M")
                timestamped_files.append((dt, f))
            except ValueError:
                # Skip files with non-matching timestamp formats
                continue

    if not timestamped_files:
        raise FileNotFoundError("No study files with a valid timestamp found in filenames.")

    # Select the file with the latest timestamp
    latest_file = max(timestamped_files, key=lambda x: x[0])[1]

    # Load the most recent study file
    study_filename = latest_file
    study = joblib.load(study_filename)
    print(f"Loaded study object from most recent file: {study_filename}")

# 1. Extract all optimal models (Pareto Front)
pareto_trials = study.best_trials

# 2. Convert the optimal trials into a DataFrame
pareto_data = []
for t in pareto_trials:
    row = {
        "trial_id": t.number,
        "Multi_Horizon_C_Index": t.values[0],
        "Brier_Score": t.values[1], # Mean time-specific Brier score
        "Global_C_Index": t.user_attrs.get("Global_C_Index", np.nan) # Extracted from user attributes
    }
    # Add the hyperparameters for this specific trial
    row.update(t.params)
    pareto_data.append(row)

df_pareto = pd.DataFrame(pareto_data)

# 3. Calculate the Distance to the Ideal Point (C-Index = 1.0, Brier Score = 0.0)
# The methodological goal is to MINIMIZE this Euclidean distance
df_pareto["Distance_to_Ideal"] = np.sqrt(
    (1.0 - df_pareto["Multi_Horizon_C_Index"])**2 + (df_pareto["Brier_Score"])**2
)

# 4. Sort to find the absolute winner (the "knee point" of the Pareto front)
df_pareto = df_pareto.sort_values("Distance_to_Ideal", ascending=True).reset_index(drop=True)

# --- RESULTS ---
print(f"\nFound {len(df_pareto)} models in the Pareto Front.")

print("\nABSOLUTE WINNER (Optimal Trade-off / Knee Point):")
winner = df_pareto.iloc[0]
print(f"  Trial ID              : {winner['trial_id']}")
print(f"  Multi-Horizon C-Index : {winner['Multi_Horizon_C_Index']:.4f}")
print(f"  Brier Score           : {winner['Brier_Score']:.4f}")
print(f"  Global C-Index        : {winner['Global_C_Index']:.4f}")
print(f"  Distance              : {winner['Distance_to_Ideal']:.4f}")

print("\nWinner Hyperparameters:")
exclude_keys = ["trial_id", "Multi_Horizon_C_Index", "Brier_Score", "Global_C_Index", "Distance_to_Ideal"]
params_winner = {k: v for k, v in winner.items() if k not in exclude_keys}
for k, v in params_winner.items():
    print(f"  {k}: {v}")

# Export the Pareto Front for backup and manuscript reporting
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
filename_pareto = f"_out/Pareto_Front_XGB_{timestamp_str}.csv"
os.makedirs("_out", exist_ok=True)
df_pareto.to_csv(filename_pareto, index=False)

print(f"\nPareto Front saved to: {filename_pareto}")

In [20]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

display(df_pareto.sort_values("Multi_Horizon_C_Index", ascending=False).head(10))

,trial_id,Multi_Horizon_C_Index,Brier_Score,Global_C_Index,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,Distance_to_Ideal
0,21,0.778628,0.01894,0.74776,0.01521,3,5,0.712648,0.444046,0.905679,0.214096,0.095706,0.222181


In [21]:
from IPython.display import display, HTML

html_content = """
<div style="font-family: Arial; line-height: 1.6;">

<h2>📊 Pareto Front Analysis</h2>

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
<thead>
<tr style="background-color:#f5f5f5;">
<th style="border:1px solid #ccc; padding:8px;">Component</th>
<th style="border:1px solid #ccc; padding:8px;">Trial 21 (Absolute Winner)</th>
<th style="border:1px solid #ccc; padding:8px;">Interpretation</th>
</tr>
</thead>
<tbody>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Multi-Horizon C-Index</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.7786</td>
<td style="border:1px solid #ccc; padding:8px;">
Trial 21 achieves strong discrimination across clinically relevant horizons (3–60 months), effectively ranking patient risk over time while handling the competing risk nuances.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Brier Score</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.0189</td>
<td style="border:1px solid #ccc; padding:8px;">
Exceptional calibration. The error rate is extremely low, meaning the predicted absolute probabilities are highly reliable and closely match the observed baseline rates.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Global C-Index</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.7478</td>
<td style="border:1px solid #ccc; padding:8px;">
Maintains robust global discrimination over the entire follow-up period, proving the model does not sacrifice overall ranking to achieve its multi-horizon calibration.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Distance to Ideal (C=1, IBS=0)</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.2222</td>
<td style="border:1px solid #ccc; padding:8px;">
This score represents the mathematical "knee point" of the Pareto front, providing an objective, non-arbitrary mathematical basis for selecting this specific trade-off.
</td>
</tr>

</tbody>
</table>

<br>

<h3>🧠 Hyperparameter Robustness Interpretation (Trial 21)</h3>

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
<thead>
<tr style="background-color:#f5f5f5;">
<th style="border:1px solid #ccc; padding:8px;">Hyperparameter</th>
<th style="border:1px solid #ccc; padding:8px;">Value</th>
<th style="border:1px solid #ccc; padding:8px;">Statistical Meaning</th>
</tr>
</thead>
<tbody>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>max_depth</b></td>
<td style="border:1px solid #ccc; padding:8px;">3</td>
<td style="border:1px solid #ccc; padding:8px;">
Consistently shallow trees reduce variance and prevent overfitting, forcing the model to rely on simple, generalizable clinical rules (additive risk) rather than memorizing complex noise.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>min_child_weight</b></td>
<td style="border:1px solid #ccc; padding:8px;">5</td>
<td style="border:1px solid #ccc; padding:8px;">
Ensures a sufficient sample size per leaf before making a split, acting as a direct safeguard against the low (~4%) mortality event rate.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>gamma</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.096</td>
<td style="border:1px solid #ccc; padding:8px;">
Minimal aggressive pruning. Because the max_depth is already severely constrained (3), the algorithm doesn't need high gamma to prevent overfitting.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>learning_rate</b></td>
<td style="border:1px solid #ccc; padding:8px;">0.015</td>
<td style="border:1px solid #ccc; padding:8px;">
A slow, highly stable optimization trajectory that prevents the gradient descent from overshooting the global minimum.
</td>
</tr>

<tr>
<td style="border:1px solid #ccc; padding:8px;"><b>Regularization (α, λ)</b></td>
<td style="border:1px solid #ccc; padding:8px;">α=0.91, λ=0.21</td>
<td style="border:1px solid #ccc; padding:8px;">
L1 penalty (α) dominates over L2. This encourages "sparsity," effectively performing automatic feature selection by dropping uninformative variables, which is optimal for linear-like survival outcomes.
</td>
</tr>

</tbody>
</table>

<p style="margin-top:15px;">
<b>Overall Interpretation:</b> Trial 21 represents a conservative, highly stable configuration optimized for a low-event-rate survival outcome. It balances a strictly shallow architecture with L1-driven feature selection, resulting in an optimal trade-off between predicting time-specific clinical risk (C-Index) and absolute probability accuracy (Brier Score).
</p>

</div>
"""

display(HTML(html_content))

Component,Trial 21 (Absolute Winner),Interpretation
Multi-Horizon C-Index,0.7786,"Trial 21 achieves strong discrimination across clinically relevant horizons (3–60 months), effectively ranking patient risk over time while handling the competing risk nuances."
Brier Score,0.0189,"Exceptional calibration. The error rate is extremely low, meaning the predicted absolute probabilities are highly reliable and closely match the observed baseline rates."
Global C-Index,0.7478,"Maintains robust global discrimination over the entire follow-up period, proving the model does not sacrifice overall ranking to achieve its multi-horizon calibration."
"Distance to Ideal (C=1, IBS=0)",0.2222,"This score represents the mathematical ""knee point"" of the Pareto front, providing an objective, non-arbitrary mathematical basis for selecting this specific trade-off."
Hyperparameter,Value,Statistical Meaning
max_depth,3,"Consistently shallow trees reduce variance and prevent overfitting, forcing the model to rely on simple, generalizable clinical rules (additive risk) rather than memorizing complex noise."
min_child_weight,5,"Ensures a sufficient sample size per leaf before making a split, acting as a direct safeguard against the low (~4%) mortality event rate."
gamma,0.096,"Minimal aggressive pruning. Because the max_depth is already severely constrained (3), the algorithm doesn't need high gamma to prevent overfitting."
learning_rate,0.015,"A slow, highly stable optimization trajectory that prevents the gradient descent from overshooting the global minimum."
"Regularization (α, λ)","α=0.91, λ=0.21","L1 penalty (α) dominates over L2. This encourages ""sparsity,"" effectively performing automatic feature selection by dropping uninformative variables, which is optimal for linear-like survival outcomes."


## Optimism correction

🔟 Take-home messages (what the code does)

- Implements Harrell’s bootstrap optimism correction.
- Uses the final tuned XGBoost Cox model (Trial 21).
- Estimates apparent C-index on full dataset.
- Determines optimal boosting rounds via early stopping.
- Trains final baseline model on 100% of data.
- Runs 100 bootstrap resamples in parallel.
- Retrains model inside each bootstrap sample.
- Computes performance on bootstrap and original data.
- Calculates optimism = apparent_boot − test_original.
- Reports optimism-corrected C-index for internal validation.

🧩 Assumptions (5 key ones)

- Bootstrap samples approximate the data-generating process.
- Model structure and hyperparameters are fixed.
- C-index is appropriate performance metric.
- IPCW assumptions hold for censoring mechanism.
- Sample size is large enough for stable bootstrap estimates.


In [35]:
# @title Harrell's Bootstrap Optimism Correction (Parallelized, CPU-2)
import numpy as np
import pandas as pd
import xgboost as xgb
import gc
import os
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sksurv.metrics import concordance_index_ipcw
from joblib import Parallel, delayed
import warnings

warnings.filterwarnings("ignore")

nb_print("Initializing Parallel Harrell's Bootstrap Optimism Correction...")

# --- CPU CONFIGURATION ---
N_CORES = max(1, os.cpu_count() - 2)
nb_print(f"Parallel Execution Configured: Using {N_CORES} CPU cores.")

# --- 1. SET TRIAL 21 HYPERPARAMETERS (ABSOLUTE WINNER) ---
params_winner = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'tree_method': 'hist',
    'device': 'cpu',
    'verbosity': 0,
    'seed': 42,
    
    # Trial 21 parameters from Optuna Pareto Front
    'learning_rate': 0.01521,
    'max_depth': 3,
    'min_child_weight': 5,
    'subsample': 0.712648,
    'colsample_bytree': 0.444046,
    'reg_alpha': 0.905679,
    'reg_lambda': 0.214096,
    'gamma': 0.095706
}

B_ITERATIONS = 500 # Standard number of bootstrap iterations for clinical papers

# --- STRATIFICATION HELPER (Safety check to ensure it exists) ---
def get_dual_stratification_labels(df, y_struct):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns: labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns: labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns: labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns: labels[df['plan_type_corr_m_pai'] == 1] = 4
    event_status = y_struct['event'].astype(int)
    return (labels * 10) + event_status

# Ensure variables are mapped correctly from your environment
strat_labels_dual = get_dual_stratification_labels(df_tune, y_tune_struct)

# --- 2. CALCULATE APPARENT PERFORMANCE ON ORIGINAL DATA ---
nb_print("Calculating apparent performance on the original full dataset...")

# Initial config uses all allocated cores for speed
params_initial = params_winner.copy()
params_initial['nthread'] = N_CORES

X_train_app, X_val_app, y_train_xgb_app, y_val_xgb_app = train_test_split(
    df_tune, y_xgb_label, test_size=0.2, random_state=42, stratify=strat_labels_dual
)

dtrain_app = xgb.DMatrix(X_train_app, label=y_train_xgb_app)
dval_app = xgb.DMatrix(X_val_app, label=y_val_xgb_app)

temp_model = xgb.train(
    params_initial, dtrain_app, 
    num_boost_round=1500, 
    evals=[(dval_app, 'val')], 
    early_stopping_rounds=30, 
    verbose_eval=False
)
optimal_boost_rounds = temp_model.best_iteration

nb_print(f"Optimal boosting rounds determined: {optimal_boost_rounds}")

# Train the definitive baseline model on 100% of the data
dorig = xgb.DMatrix(df_tune, label=y_xgb_label)
baseline_model = xgb.train(
    params_initial, dorig, 
    num_boost_round=optimal_boost_rounds,
    verbose_eval=False
)

risk_orig = baseline_model.predict(dorig)
try:
    c_apparent_orig = concordance_index_ipcw(y_tune_struct, y_tune_struct, risk_orig)[0]
except:
    from sksurv.metrics import concordance_index_censored
    c_apparent_orig = concordance_index_censored(y_tune_struct['event'], y_tune_struct['time'], risk_orig)[0]

nb_print(f"Baseline Apparent C-index: {c_apparent_orig:.4f}")

# --- 3. PARALLEL BOOTSTRAP WORKER FUNCTION ---
def parallel_bootstrap_worker(b, df_original, y_xgb_original, y_struct_orig, params, opt_rounds):
    # CRITICAL: Force 1 thread per XGBoost model to prevent CPU thrashing during multiprocessing
    boot_params = params.copy()
    boot_params['nthread'] = 1 
    
    indices = np.arange(len(df_original))
    boot_indices = resample(indices, replace=True, n_samples=len(indices), random_state=b)
    
    X_boot = df_original.iloc[boot_indices]
    y_xgb_boot = y_xgb_original[boot_indices]
    y_struct_boot = y_struct_orig[boot_indices]
    
    # Recreate DMatrix objects strictly inside the isolated worker
    dboot = xgb.DMatrix(X_boot, label=y_xgb_boot)
    dorig_local = xgb.DMatrix(df_original, label=y_xgb_original)
    
    boot_model = xgb.train(
        boot_params, dboot, 
        num_boost_round=opt_rounds,
        verbose_eval=False
    )
    
    # Evaluate on Bootstrap Sample (Apparent Boot Performance)
    risk_boot = boot_model.predict(dboot)
    try:
        c_boot_app = concordance_index_ipcw(y_struct_boot, y_struct_boot, risk_boot)[0]
    except:
        from sksurv.metrics import concordance_index_censored
        c_boot_app = concordance_index_censored(y_struct_boot['event'], y_struct_boot['time'], risk_boot)[0]
        
    # Evaluate on Original Data (Test Performance)
    risk_test_orig = boot_model.predict(dorig_local)
    try:
        c_boot_test = concordance_index_ipcw(y_struct_boot, y_struct_orig, risk_test_orig)[0]
    except:
        from sksurv.metrics import concordance_index_censored
        c_boot_test = concordance_index_censored(y_struct_orig['event'], y_struct_orig['time'], risk_test_orig)[0]
        
    # Calculate Optimism
    optimism = c_boot_app - c_boot_test
    return optimism

# --- 4. EXECUTE PARALLEL BOOTSTRAP LOOP ---
nb_print(f"\nLaunching {B_ITERATIONS} Parallel Bootstrap Iterations...")

optimism_values = Parallel(n_jobs=N_CORES, verbose=10)(
    delayed(parallel_bootstrap_worker)(
        b, df_tune, y_xgb_label, y_tune_struct, params_winner, optimal_boost_rounds
    ) for b in range(B_ITERATIONS)
)

# --- 5. CALCULATE FINAL CORRECTED METRICS ---
mean_optimism = np.mean(optimism_values)
c_index_corrected = c_apparent_orig - mean_optimism

nb_print("\n--------------------------------------------------")
nb_print("FINAL OPTIMISM-CORRECTED RESULTS")
nb_print("--------------------------------------------------")
nb_print(f"Apparent C-Index (Original Data) : {c_apparent_orig:.4f}")
nb_print(f"Mean Optimism (from {B_ITERATIONS} boots)   : {mean_optimism:.4f}")
nb_print(f"Optimism-Corrected C-Index       : {c_index_corrected:.4f}")
nb_print("--------------------------------------------------")

# Save results to CSV for documentation
os.makedirs("_out", exist_ok=True)
results_df = pd.DataFrame({
    'Metric': ['Apparent_C_Index', 'Mean_Optimism', 'Corrected_C_Index'],
    'Value': [c_apparent_orig, mean_optimism, c_index_corrected]
})
timestamp_str = pd.Timestamp.now().strftime("%Y%m%d_%H%M")
filename = f"_out/XGB_Death_Bootstrap_Optimism_Results_{timestamp_str}.csv"
results_df.to_csv(filename, index=False)
nb_print(f"Results saved successfully to {filename}.")

[Parallel(n_jobs=30)]: Using backend LokyBackend with 30 concurrent workers.
[Parallel(n_jobs=30)]: Done   1 tasks      | elapsed:   31.0s
[Parallel(n_jobs=30)]: Done  12 tasks      | elapsed:   35.4s
[Parallel(n_jobs=30)]: Done  25 tasks      | elapsed:   38.7s
[Parallel(n_jobs=30)]: Done  38 tasks      | elapsed:  1.0min
[Parallel(n_jobs=30)]: Done  53 tasks      | elapsed:  1.2min
[Parallel(n_jobs=30)]: Done  68 tasks      | elapsed:  1.5min
[Parallel(n_jobs=30)]: Done  85 tasks      | elapsed:  1.7min
[Parallel(n_jobs=30)]: Done 102 tasks      | elapsed:  2.2min
[Parallel(n_jobs=30)]: Done 121 tasks      | elapsed:  2.3min
[Parallel(n_jobs=30)]: Done 140 tasks      | elapsed:  2.8min
[Parallel(n_jobs=30)]: Done 161 tasks      | elapsed:  3.3min
[Parallel(n_jobs=30)]: Done 182 tasks      | elapsed:  3.4min
[Parallel(n_jobs=30)]: Done 205 tasks      | elapsed:  3.9min
[Parallel(n_jobs=30)]: Done 228 tasks      | elapsed:  4.4min
[Parallel(n_jobs=30)]: Done 253 tasks      | elapsed:  

~ 9 minutes

In [37]:
import pandas as pd
from IPython.display import HTML, display

# Example: limit rows/columns shown in console
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

# Convert DataFrame to HTML and wrap in a scrollable div
html_table = df_results.to_html()
scroll_box = f"""
<div style="max-height:400px; max-width:1000px; overflow-y:auto; overflow-x:auto; border:1px solid #ccc;">
{html_table}
</div>
"""
display(HTML(scroll_box))


,subsample,reg_lambda,reg_alpha,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree,objective,eval_metric,tree_method,seed,nthread,device,verbosity,Unos_C_Index,Std_Dev
16,0.9,0.1,0.1,1,5,0.010,0.1,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.745007,0.017049
41,0.9,1.0,1.0,10,3,0.010,0.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744481,0.016503
44,0.7,1.0,0.1,10,4,0.010,0.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744270,0.015859
32,0.6,1.0,0.0,1,4,0.010,0.1,0.8,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744245,0.015812
20,0.7,5.0,5.0,10,4,0.005,0.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.744180,0.015198
43,0.8,0.1,0.1,10,3,0.010,0.0,0.8,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743670,0.015671
12,0.9,1.0,0.1,10,3,0.005,2.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743611,0.016138
17,0.8,5.0,10.0,1,5,0.010,1.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743482,0.015742
7,0.8,1.0,10.0,20,8,0.010,2.0,0.5,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743223,0.016286
19,0.6,0.1,0.1,20,8,0.005,0.0,0.6,survival:cox,cox-nloglik,hist,42,30,cpu,0,0.743158,0.015927


In [38]:
from IPython.display import display, HTML

html_content = """
<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6; color: #333; max-width: 850px;">

<h2 style="color: #2c3e50; border-bottom: 2px solid #ecf0f1; padding-bottom: 5px;">🧠 Decoding the XGBoost Grid Search</h2>
<p style="font-size: 15px;">
We tested 50 different configurations to see how XGBoost learns best from the mortality data. 
By looking at what the top models share (and what the bottom models did wrong), the data tells a very clear story about the underlying biology of the patients.
</p>

<table style="width: 100%; border-collapse: collapse; margin-top: 20px;">
    <tr>
        <td style="width: 50%; vertical-align: top; padding-right: 15px;">
            <h3 style="color: #2980b9;">🐢 1. "Slow and Steady" Wins</h3>
            <p style="font-size: 14px;">
                <b>Learning Rate:</b> If you look at the top 15 models, they strictly use low learning rates (<code>0.010</code> or <code>0.005</code>). The models at the very bottom of the table rushed the process with <code>0.100</code>.<br>
                <i>Meaning:</i> The algorithm must take tiny, careful steps to find true mortality risks without tripping over random noise.
            </p>
        </td>
        <td style="width: 50%; vertical-align: top; padding-left: 15px; border-left: 1px solid #eee;">
            <h3 style="color: #2980b9;">🌳 2. Shallow over Deep</h3>
            <p style="font-size: 14px;">
                <b>Max Depth:</b> The best performing trees are very shallow (Depth <code>3</code>, <code>4</code>, or <code>5</code>). Whenever we allowed deep, complex trees (Depth <code>8</code>, seen at the bottom), performance dropped.<br>
                <i>Meaning:</i> Mortality risk is additive and straightforward. Complex, deep decision branches just memorize random patient outliers (overfitting).
            </p>
        </td>
    </tr>
    <tr>
        <td style="width: 50%; vertical-align: top; padding-right: 15px; padding-top: 15px;">
            <h3 style="color: #2980b9;">🛡️ 3. Guardrails for Rare Events</h3>
            <p style="font-size: 14px;">
                <b>Min Child Weight:</b> Many top models favor values like <code>10</code> or <code>20</code>.<br>
                <i>Meaning:</i> This forces the tree to only create a new clinical "rule" if it applies to a solid group of patients, preventing wild guesses based on 1 or 2 isolated deaths.
            </p>
        </td>
        <td style="width: 50%; vertical-align: top; padding-left: 15px; padding-top: 15px; border-left: 1px solid #eee;">
            <h3 style="color: #2980b9;">✂️ 4. Mathematical Penalties</h3>
            <p style="font-size: 14px;">
                <b>Regularization (Alpha/Lambda):</b> The presence of L1 and L2 penalties in the top ranks shows that XGBoost performs better when it is actively punished for adding unnecessary variables.
            </p>
        </td>
    </tr>
</table>

<div style="background-color: #f8f9fa; border-left: 5px solid #27ae60; padding: 15px; margin-top: 25px; border-radius: 0 5px 5px 0;">
    <h4 style="margin-top: 0; color: #27ae60; font-size: 16px;">💡 The Clinical Takeaway</h4>
    <p style="margin-bottom: 0; font-size: 15px;">
        This table proves computationally what we suspected biologically: mortality in this population doesn't have "secret, complex formulas". The risk is driven by strong, direct factors. XGBoost reached high discrimination by acting almost like a traditional linear model—moving slowly, keeping rules simple, and aggressively filtering out noise.
    </p>
</div>

<!-- New compact results box added with minimal change to original layout -->
<div style="background:#fff7e6; border-left:5px solid #f39c12; padding:12px; margin-top:18px; border-radius:4px; max-width:850px;">
  <strong style="color:#d35400;">Recent internal validation update</strong>
  <ul style="margin:8px 0 0 18px; font-size:14px; color:#333;">
    <li>Final tuned model used for validation: <b>Trial 21</b>.</li>
    <li>Optimal boosting rounds (early stopping): <b>708</b>.</li>
    <li>Apparent C-index on full dataset: <b>0.7672</b>.</li>
    <li>Harrell's bootstrap: <b>500</b> resamples run in parallel using <b>30</b> CPU cores.</li>
    <li>Mean optimism (500 boots): <b>0.0184</b>.</li>
    <li>Optimism-corrected C-index (internal validation): <b>0.7488</b>.</li>
    <li>Results saved to: <b>_out/XGB_Death_Bootstrap_Optimism_Results_20260223_1154.csv</b>.</li>
  </ul>
</div>

</div>
"""

display(HTML(html_content))


"🐢 1. ""Slow and Steady"" Wins Learning Rate: If you look at the top 15 models, they strictly use low learning rates (0.010 or 0.005). The models at the very bottom of the table rushed the process with 0.100. Meaning: The algorithm must take tiny, careful steps to find true mortality risks without tripping over random noise.","🌳 2. Shallow over Deep Max Depth: The best performing trees are very shallow (Depth 3, 4, or 5). Whenever we allowed deep, complex trees (Depth 8, seen at the bottom), performance dropped. Meaning: Mortality risk is additive and straightforward. Complex, deep decision branches just memorize random patient outliers (overfitting)."
"🛡️ 3. Guardrails for Rare Events Min Child Weight: Many top models favor values like 10 or 20. Meaning: This forces the tree to only create a new clinical ""rule"" if it applies to a solid group of patients, preventing wild guesses based on 1 or 2 isolated deaths.",✂️ 4. Mathematical Penalties Regularization (Alpha/Lambda): The presence of L1 and L2 penalties in the top ranks shows that XGBoost performs better when it is actively punished for adding unnecessary variables.
